In [1]:
from pathlib import Path
import os, sys
from dotenv import load_dotenv

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_score_providers import seed_score_providers
from app.db.seeders.seed_tool_providers import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_state_providers import seed_state_providers
from app.db.seeders.seed_system_providers import seed_system_providers
from app.db.seeders.seed_controller_providers import seed_controller_providers
from app.db.seeders.seed_program_provider import seed_program_providers
from app.db.seeders.seed_session_configs import seed_session_configs
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_state_providers(session)
    seed_system_providers(session)
    seed_controller_providers(session)
    seed_program_providers(session)
    seed_session_configs(session)

# Load from env/.env relative to project root
dotenv_path = Path("env/.env").resolve()
if dotenv_path.exists():
    load_dotenv(dotenv_path)
    print("✅ Loaded environment variables from env/.env")
else:
    raise FileNotFoundError(f"❌ Missing .env file at: {dotenv_path}")

Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt 'generate' with ID: 1 and GUID: e3a9e4fb-b405-4f10-b992-6ddd4702c524
Seeded AgentPrompt 'linting_generator_agent' with ID: 2 and GUID: a2e69c44-498b-4820-8035-4906f62f46ae
Seeded SystemPrompt 'format' with ID: 1 and GUID: d267d1ee-ffa6-4668-9580-b6392c9fd6ba
Seeded SystemPrompt 'linting_system' with ID: 2 and GUID: 5c372716-991a-4589-83b0-b65003cd5828
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.
✅ Seeded state provider configurations successfully.
Seeded system provider configurations successfully.
✅ Seeded controller providers
✅ Seeded program providers
✅ Seeded session configs
✅ Loaded environment variables from env/.env


In [3]:
import sqlite3
import json
from app.factories.tool_provider_factory import ToolProviderFactory
from app.db.connection import get_connection

# 1. Run the tool
tool = ToolProviderFactory.create(2)  # Adjust ID to match sonarcloud
session_id = "sonar-log-test"
result = tool.run({"target": "tests/example.py"}, session_id=session_id)
print("📦 Tool output:", result)

# 2. Verify log contents in provider_log table
conn = get_connection()
cursor = conn.cursor()

print("\n🔍 Logged entries for session:", session_id)
for row in cursor.execute("SELECT provider_type, input, output FROM provider_log WHERE session_id = ?", (session_id,)):
    provider_type, raw_input, raw_output = row
    print(f"\nType: {provider_type}")

    input_dict = json.loads(raw_input)
    output_dict = json.loads(json.loads(raw_output))  # <- this is the fix

    print("🔹 input keys:", list(input_dict.keys()))
    print("🔸 metrics:", output_dict.get("metrics"))

conn.close()


📄 Target file: example.py
🔐 Environment variables loaded
📦 Cloning repo into C:\Users\ben\AppData\Local\Temp\tmp1a9n01_8
📤 Pushed file to remote repository
⏳ Waiting for SonarCloud analysis to complete...
🔎 Scan status check 1/20
📡 Scan status: SUCCESS
📊 Full metrics poll attempt 1/10
{"component":{"id":"AZcjOrI0fsiRa2-UqmYy","key":"benpodraza_codecritic_scoring","name":"codecritic_scoring","qualifier":"TRK","measures":[{"metric":"coverage","value":"0.0","bestValue":false},{"metric":"complexity","value":"10"},{"metric":"code_smells","value":"4","bestValue":false},{"metric":"duplicated_lines_density","value":"0.0","bestValue":true},{"metric":"duplicated_lines","value":"0","bestValue":true},{"metric":"functions","value":"4"},{"metric":"classes","value":"0"},{"metric":"statements","value":"38"},{"metric":"security_hotspots","value":"0","bestValue":true},{"metric":"bugs","value":"1","bestValue":false},{"metric":"lines_to_cover","value":"38"},{"metric":"cognitive_complexity","value":"14","b